In [4]:
import cadquery as cq
from jupyter_cadquery.cadquery import show
from jupyter_cadquery import set_sidecar

set_sidecar("Test", init=True)
objects = []

def show_object(obj):
    global objects
    objects.append(obj)
    show(*objects)

Overwriting auto display for cadquery Workplane and Shape


In [5]:
import math

## Brushless Settings

In [3]:
mount_plate_thickness = 0.005*1000
engine_screw_hole_circle = 0.042*1000
engine_mount_box_length = 0.0133 * 2.5*1000
engine_down_thrust_deg: float = -2.5
engine_side_thrust_deg: float = -2.5
engine_screw_din_diameter = 0.0032*1000
engine_full_cover_length = 0.0452*1000
engine_pos = (0,0,0)

In [4]:
from math import sqrt
mount = cq.Workplane("YZ", origin=engine_pos).workplane(offset=engine_full_cover_length+engine_mount_box_length).tag('front')\
.rect(sqrt(0.5)*engine_screw_hole_circle,sqrt(0.5)*engine_screw_hole_circle,forConstruction=True).last()\
.vertices().tag('corners').cylinder(engine_mount_box_length*1,(engine_screw_din_diameter+6)/2).faces("<X").tag('cyl')\
.vertices(tag='corners').cylinder(engine_mount_box_length*1,(engine_screw_din_diameter+6)/2).faces("<X")\
.vertices(tag='corners').cylinder(engine_mount_box_length*2,(engine_screw_din_diameter)/2, combine='cut')\
.rotate(axisStartPoint=engine_pos, axisEndPoint=(engine_pos[0], engine_pos[1], engine_pos[2]+1), angleDegrees=engine_side_thrust_deg)\
.rotate(axisStartPoint=engine_pos, axisEndPoint=(engine_pos[0], engine_pos[1]+1, engine_pos[2]), angleDegrees=engine_down_thrust_deg)\
.faces(tag='cyl').rect(engine_screw_hole_circle*10,engine_screw_hole_circle*10).extrude(-engine_mount_box_length, combine='cut')\
.workplaneFromTagged('front').box(engine_screw_hole_circle, engine_screw_hole_circle, engine_mount_box_length)\
#.faces(">X").tag('rear').rect(engine_screw_hole_circle * 0.7, engine_screw_hole_circle * 0.7).cutBlind(-engine_mount_box_length * 1.2)
#.faces(tag='rear').workplane().rect(engine_screw_hole_circle*10,engine_screw_hole_circle*10).extrude(engine_mount_box_length, combine='cut')
#.workplaneFromTagged('rotated_wp').translate((-engine_mount_box_length/2,0,0))
mount

In [5]:
mount = box.translate((52+engine_mount_box_length/2,0,0))
mount

NameError: name 'box' is not defined

In [ ]:
mount.rotate((0,0,0),(0,0,1),180)

In [ ]:
cq.Workplane("YZ").box(1,1,0.2).faces("<X or >X").shell(0.1)

In [ ]:
ass = cq.Assembly(obj=mount, color=cq.Color('red'), name="horst")
ass.add(cq.Workplane('XY').box(20,20,20), name="green")

# Special 

In [6]:
def step_importer(path_) -> cq.Workplane:
    from cadquery import importers
    shapes = importers.importStep(path_)
    return shapes

In [7]:
from OCP.BRepOffsetAPI import BRepOffsetAPI_MakeOffsetShape
def offset3D(self: cq.Workplane, offset: float) -> cq.Workplane:
    solid = self.findSolid()
    maker = BRepOffsetAPI_MakeOffsetShape()
    maker.PerformBySimple(solid.wrapped, offset)
    of_shape = maker.Shape()
    new_solid = cq.Solid(of_shape)
    return self.newObject([new_solid])

cq.Workplane.offset3D = offset3D

## Fuselage

In [11]:
fuselage = step_importer("./fuselage_inlets.step").findSolid().scale(40); fuselage

In [8]:
wing = step_importer("./wing.step").findSolid().scale(40); wing

In [119]:
ass = cq.Assembly(obj=fuselage, name="fuselage")
ass.add(wing, name="wing")

100% ⋮————————————————————————————————————————————————————————————⋮ (2/2) 10.61s


In [134]:
fbbox = fuselage.BoundingBox()
wbbox = wing.BoundingBox()

In [154]:
cq.Workplane(inPlane="XZ", obj=wing).tag('middle').workplane(offset=fbbox.ymax).split(keepBottom=True)\
.workplaneFromTagged('middle').workplane(offset=fbbox.ymin).split(keepTop=True)


In [142]:
fbbox.ymax

83.43252949253545

In [164]:
rect = cq.Workplane('YZ').rect(10,10)


In [171]:
rect.edges('<Z').cylinder(radius=9, height=20,direct=(0,1,0))

In [173]:
cq.Workplane('YZ').center(0,10)

100% ⋮————————————————————————————————————————————————————————————⋮ (2/2)  0.00s


In [194]:
cq.Workplane('XZ').move(10,0).cylinder(100,10)

In [209]:
cq.Workplane('XY').sketch().trapezoid(100,20,45).finalize().extrude(10)


In [13]:
type(fuselage)

cadquery.occ_impl.shapes.Compound

In [12]:
fuselage_small = cq.CQ(fuselage).offset3D(-0.8)


In [15]:
shell = cq.CQ(fuselage).cut(fuselage_small)

In [19]:
fus = cq.CQ(fuselage)

In [25]:
cq.Workplane().box(1000,200,200).cut(fuselage)

## wing

In [5]:
wing = step_importer("./wing.step").findSolid().scale(40); wing